# 🧹 Data Cleaning Pipeline for Electrical Products Dataset

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import stats

# Load dataset
df = pd.read_csv("electrical_products.csv")
print(df)


   product_id  product_name     category        brand  model_year  \
0        P000   Product_0 😎   appliance    ElectroMax        2021   
1        P001  Product_1 :)        MOTOR          NaN        2016   
2        P002   Product_2 💡     Lighting    PowerLite        2019   
3        P003   Product_3 ♥      sensor     SparkTech        2017   
4        P004     Product_4       Sensor    SparkTech        2020   
5        P005     Product_5        Motor    PowerLite        2018   
6        P006     Product_6      Battery  GreenCharge        2016   
7        P007     Product_7    Appliance          NaN        2021   
8        P008     Product_8     Lighting  GreenCharge        2022   
9        P009     Product_9     Lighting    VoltCraft        2024   
10       P010    Product_10       Sensor    SparkTech        2020   
11       P011    Product_11    Appliance    SparkTech        2020   
12       P012    Product_12    Appliance          NaN        2015   
13       P013    Product_13     Li

## 1 Imputation / Deletion for Missing Values

In [2]:
# Show missing values
print(df.isnull().sum())

# Impute categorical with mode, numerical with median
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns

df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print("✅ Missing values handled.")


product_id                 0
product_name               0
category                   0
brand                      4
model_year                 0
warranty_period            0
voltage_rating             0
current_rating             0
power_rating_watts         0
frequency_hz               0
efficiency_rating          3
power_factor               0
number_of_phases           0
battery_operated           0
battery_type               7
energy_star_certified      0
standby_power_watts        0
average_daily_use_hours    0
weight_kg                  0
dimensions_cm              0
volume_cm3                 0
casing_material            0
color                      0
ip_rating                  0
thermal_resistance         0
control_type               0
smart_device               0
connectivity               6
number_of_ports            0
sensor_type                2
price_usd                  5
discount_percentage        0
stock_quantity             0
units_sold                 0
return_rate   

## 2️⃣ Removing Duplicates

In [3]:
df = df.drop_duplicates()
print("✅ Duplicates removed.")


✅ Duplicates removed.


## 3️⃣ Standardizing Categorical Labels

In [4]:
df['efficiency_rating'] = df['efficiency_rating'].str.strip().str.upper().str.replace(" ", "")
df['category'] = df['category'].str.strip().str.capitalize()
print(df['efficiency_rating'].unique())
print(df['category'].unique())


['A++' 'A+' 'A']
['Appliance' 'Motor' 'Lighting' 'Sensor' 'Battery']


## 4️⃣ Fixing Price Format and Converting to Float

In [ ]:
df['price_usd'] = df['price_usd'].astype(str).str.replace('$', '', regex=False)
df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
df['price_usd'] = df['price_usd'].fillna(df['price_usd'].median())


## 5️⃣ Detecting & Removing Outliers (Z-score Method)

In [ ]:
z_scores = np.abs(stats.zscore(df.select_dtypes(include=[np.number])))
df = df[(z_scores < 3).all(axis=1)]
print("✅ Outliers removed.")


## 6️⃣ Removing Emojis & Emoticons from Text Fields

In [ ]:
def remove_emojis(text):
    return re.sub(r'[^ -]+', '', text)

df['product_name'] = df['product_name'].apply(remove_emojis)


## 7️⃣ Scaling Numeric Fields

In [ ]:
scaler = MinMaxScaler()
scaled_cols = ['price_usd', 'power_rating_watts', 'weight_kg']
df[scaled_cols] = scaler.fit_transform(df[scaled_cols])


## 8️⃣ Encoding Categorical Fields

In [ ]:
categorical_cols = ['category', 'brand', 'efficiency_rating', 'battery_type', 'control_type']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


## 9️⃣ Feature Engineering

In [ ]:
# Create price_per_kg and usage_score
df['price_per_kg'] = df['price_usd'] / (df['weight_kg'] + 0.01)
df['usage_score'] = df['average_daily_use_hours'] * df['smart_score']
